# MCP Data Analyst — Tier 2 Test Notebook
## Interactive test of medium-level data tools

This notebook tests every Tier 2 tool by calling `engine.py` directly,
simulating how an MCP client would interact with the server.

In [ ]:
import sys
import shutil
from pathlib import Path

# Ensure project root is on the path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from servers.data_medium.engine import (
    check_outliers,
    scan_nulls_zeros,
    enrich_with_geo,
    validate_dataset,
    compute_aggregations,
    run_cleaning_pipeline,
)
from servers.data_basic.engine import apply_patch, load_dataset

import pandas as pd

## 00 — Setup test fixtures

In [ ]:
WORK = Path("test_workspace_t2")
WORK.mkdir(exist_ok=True)

# --- Dataset with outliers and nulls ---
OUTLIER_CSV = WORK / "outlier_data.csv"
OUTLIER_CSV.write_text("""Region,Revenue,Units,Discount
West,5000,10,0.1
West,4800,9,0.12
East,7500,15,0.05
South,2100,5,0.2
North,4800,12,0.08
West,5200,11,0.1
East,7200,14,0.06
South,1900,4,0.25
North,5100,13,0.07
West,500,1,0.3
East,50000,100,0.01
South,,3,0.15
North,4900,11,0.09
West,5100,10,0.11
""")

# --- Clean dataset for aggregations ---
AGG_CSV = WORK / "sales_agg.csv"
AGG_CSV.write_text("""Region,Product,Revenue,Units_Sold,Quarter
West,Widget A,5000,10,Q1
West,Widget B,3200,8,Q1
East,Widget A,7500,15,Q2
South,Widget C,2100,5,Q2
North,Widget A,4800,12,Q3
West,Widget A,6000,12,Q3
East,Widget B,3000,7,Q4
South,Widget A,2500,6,Q4
North,Widget C,1800,4,Q1
West,Widget C,4200,9,Q2
""")

# --- GeoJSON fixture ---
GEO_JSON = WORK / "states.geojson"
GEO_JSON.write_text("""{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {"name": "California", "code": "CA"},
      "geometry": {"type": "Polygon", "coordinates": [[[-120,35],[-120,40],[-115,40],[-115,35],[-120,35]]]}
    },
    {
      "type": "Feature",
      "properties": {"name": "Texas", "code": "TX"},
      "geometry": {"type": "Polygon", "coordinates": [[[-105,26],[-105,36],[-95,36],[-95,26],[-105,26]]]}
    },
    {
      "type": "Feature",
      "properties": {"name": "New York", "code": "NY"},
      "geometry": {"type": "Polygon", "coordinates": [[[-80,40],[-80,45],[-72,45],[-72,40],[-80,40]]]}
    }
  ]
}""")

# --- CSV for geo enrichment ---
GEO_CSV = WORK / "geo_data.csv"
GEO_CSV.write_text("""State,Population
California,39500000
Texas,29000000
New York,19450000
""")

print(f"Workspace: {WORK.absolute()}")
print(f"Files created: outlier_data.csv, sales_agg.csv, states.geojson, geo_data.csv")

## 01 — check_outliers

In [ ]:
# Scan all numeric columns with both methods
r = check_outliers(str(OUTLIER_CSV))
assert r["success"] is True
print(f"Method: {r['method']}")
print(f"Scanned columns: {r['scanned_columns']}")
print(f"Columns with outliers: {r['columns_with_outliers']}")
for col, data in r["results"].items():
    print(f"  {col}: {data}")
print("PASS: check_outliers — both methods")

In [ ]:
# IQR method only
r = check_outliers(str(OUTLIER_CSV), method="iqr")
assert "has_outliers_std" not in r["results"]["Revenue"]
print(f"IQR results for Revenue: {r['results']['Revenue']}")
print("PASS: check_outliers — IQR only")

In [ ]:
# Specific columns
r = check_outliers(str(OUTLIER_CSV), columns=["Revenue"])
assert len(r["results"]) == 1
assert "Revenue" in r["results"]
print(f"Revenue outliers: {r['results']['Revenue']}")
print("PASS: check_outliers — specific columns")

In [ ]:
# Column not found
r = check_outliers(str(OUTLIER_CSV), columns=["NonExistent"])
assert r["success"] is False
print(f"Error: {r['error']}")
print("PASS: check_outliers — column not found")

## 02 — scan_nulls_zeros

In [ ]:
# Full scan with zeros
r = scan_nulls_zeros(str(OUTLIER_CSV))
assert r["success"] is True
print(f"Total rows: {r['total_rows']}")
print(f"Clean columns: {r['clean_columns']}")
print(f"Flagged columns: {r['flagged_columns']}")
for col, data in r["results"].items():
    print(f"  {col}: {data}")
print(f"Suggested actions: {r['suggested_actions']}")
print("PASS: scan_nulls_zeros — full scan")

In [ ]:
# Exclude zeros
r = scan_nulls_zeros(str(OUTLIER_CSV), include_zeros=False)
for col, data in r["results"].items():
    assert "zero_count" not in data or data["zero_count"] is None
print("PASS: scan_nulls_zeros — include_zeros=False")

In [ ]:
# Clean dataset — no issues
r = scan_nulls_zeros(str(AGG_CSV))
assert r["flagged_columns"] == 0
assert r["results"] == {}
print(f"Flagged: {r['flagged_columns']}")
print("PASS: scan_nulls_zeros — clean dataset")

## 03 — validate_dataset

In [ ]:
# Validate messy dataset
r = validate_dataset(str(OUTLIER_CSV))
assert r["success"] is True
print(f"Passed: {r['passed']}")
print(f"Score: {r['score']}/100")
print(f"Duplicate count: {r['duplicate_count']}")
print(f"Null summary: {r['null_summary']}")
for issue in r["issues"]:
    print(f"  [{issue['severity']}] {issue['column']}: {issue['issue']}")
print("PASS: validate_dataset — messy data")

In [ ]:
# Validate clean dataset
r = validate_dataset(str(AGG_CSV))
assert r["passed"] is True
assert r["score"] == 100
print(f"Score: {r['score']}/100, Passed: {r['passed']}")
print("PASS: validate_dataset — clean data")

In [ ]:
# With expected dtypes
r = validate_dataset(str(AGG_CSV), expected_dtypes={"Revenue": "float64"})
assert "dtype_mismatches" in r
print(f"Dtype mismatches: {r['dtype_mismatches']}")
print("PASS: validate_dataset — dtype check")

In [ ]:
# Skip duplicate check
r = validate_dataset(str(OUTLIER_CSV), check_duplicates=False)
for issue in r["issues"]:
    assert "duplicate" not in issue["issue"].lower()
print("PASS: validate_dataset — skip duplicates")

## 04 — compute_aggregations

In [ ]:
# Sum by Region
r = compute_aggregations(str(AGG_CSV), group_by=["Region"], agg_column="Revenue", agg_func="sum")
assert r["success"] is True
print(f"Group by: {r['group_by']}")
print(f"Agg: {r['agg_func']} of {r['agg_column']}")
print(f"Groups returned: {r['returned']}")
for row in r["result"]:
    print(f"  {row}")
print("PASS: compute_aggregations — sum by region")

In [ ]:
# Mean by Region
r = compute_aggregations(str(AGG_CSV), group_by=["Region"], agg_column="Revenue", agg_func="mean")
print(f"Mean revenue by region:")
for row in r["result"]:
    print(f"  {row}")
print("PASS: compute_aggregations — mean")

In [ ]:
# Count by Product
r = compute_aggregations(str(AGG_CSV), group_by=["Product"], agg_column="Revenue", agg_func="count")
print(f"Count by product:")
for row in r["result"]:
    print(f"  {row}")
print("PASS: compute_aggregations — count")

In [ ]:
# Top N
r = compute_aggregations(str(AGG_CSV), group_by=["Region"], agg_column="Revenue", agg_func="sum", top_n=2)
assert len(r["result"]) == 2
print(f"Top 2 regions by revenue:")
for row in r["result"]:
    print(f"  {row}")
print("PASS: compute_aggregations — top_n")

In [ ]:
# Multi-column group-by
r = compute_aggregations(str(AGG_CSV), group_by=["Region", "Product"], agg_column="Revenue", agg_func="sum")
print(f"Multi-group results ({r['returned']} rows):")
for row in r["result"]:
    print(f"  {row}")
print("PASS: compute_aggregations — multi-column group-by")

In [ ]:
# Invalid agg_func
r = compute_aggregations(str(AGG_CSV), group_by=["Region"], agg_column="Revenue", agg_func="median")
assert r["success"] is False
print(f"Error: {r['error']}")
print(f"Hint: {r['hint']}")
print("PASS: compute_aggregations — invalid agg_func")

## 05 — enrich_with_geo

In [ ]:
# Dry run
r = enrich_with_geo(
    str(GEO_CSV),
    str(GEO_JSON),
    join_column="State",
    geo_join_column="name",
    dry_run=True
)
if not r["success"]:
    print(f"SKIPPED: {r['error']}")
    print(f"Hint: {r.get('hint', '')}")
else:
    assert r["dry_run"] is True
    print(f"Matched: {r['matched']}")
    print(f"New columns: {r['new_columns']}")
    print("PASS: enrich_with_geo — dry_run")

In [ ]:
# Full enrichment
r = enrich_with_geo(
    str(GEO_CSV),
    str(GEO_JSON),
    join_column="State",
    geo_join_column="name"
)
if not r["success"]:
    print(f"SKIPPED: {r['error']}")
else:
    print(f"Matched: {r['matched']}")
    print(f"Output: {r['output_file']}")
    df = pd.read_csv(str(GEO_CSV))
    print(f"Enriched columns: {list(df.columns)}")
    print("PASS: enrich_with_geo — full enrichment")

## 06 — run_cleaning_pipeline

In [ ]:
# Reset outlier data for pipeline test
OUTLIER_CSV.write_text("""Region,Revenue,Units,Discount
West,5000,10,0.1
west,500,1,0.3
East,7500,15,0.05
South,,5,0.2
North,4800,12,0.08
West,5000,10,0.1
""")

# Dry run
r = run_cleaning_pipeline(str(OUTLIER_CSV), [
    {"op": "clean_text", "scope": "both"},
    {"op": "fill_nulls", "column": "Revenue", "strategy": "median"},
    {"op": "drop_duplicates"},
], dry_run=True)
assert r["success"] is True
assert r["dry_run"] is True
print(f"Would apply {r['total_ops']} ops:")
for wc in r["would_change"]:
    print(f"  - {wc['op']}")
print("PASS: run_cleaning_pipeline — dry_run")

In [ ]:
# Reset and run full pipeline
OUTLIER_CSV.write_text("""Region,Revenue,Units,Discount
West,5000,10,0.1
west,500,1,0.3
East,7500,15,0.05
South,,5,0.2
North,4800,12,0.08
West,5000,10,0.1
""")

r = run_cleaning_pipeline(str(OUTLIER_CSV), [
    {"op": "clean_text", "scope": "both"},
    {"op": "fill_nulls", "column": "Revenue", "strategy": "median"},
    {"op": "drop_duplicates"},
])
assert r["success"] is True
print(f"Applied {r['applied']}/{r['total_ops']} ops")
for s in r["summary"]:
    print(f"  {s['op']}: {s}")

df = pd.read_csv(str(OUTLIER_CSV))
print(f"Final shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("PASS: run_cleaning_pipeline — full run")

In [ ]:
# Empty ops list
r = run_cleaning_pipeline(str(OUTLIER_CSV), [])
assert r["success"] is False
print(f"Error: {r['error']}")
print("PASS: run_cleaning_pipeline — empty ops")

In [ ]:
# Pipeline with failure — should rollback
OUTLIER_CSV.write_text("""Region,Revenue,Units,Discount
West,5000,10,0.1
East,7500,15,0.05
South,2100,5,0.2
""")
before = OUTLIER_CSV.read_text()

r = run_cleaning_pipeline(str(OUTLIER_CSV), [
    {"op": "clean_text", "scope": "both"},
    {"op": "explode_table"},  # invalid op
])
assert r["success"] is False
print(f"Failed at op {r['applied']}: {r['error']}")
print("PASS: run_cleaning_pipeline — failure rollback")

## 07 — Full Tier 2 Workflow: Scan → Validate → Clean → Aggregate

In [ ]:
# Create a fresh messy dataset for the full workflow
WORKFLOW_CSV = WORK / "workflow_data.csv"
WORKFLOW_CSV.write_text("""Region,Product,Revenue,Units,Discount
West,Widget A,5000,10,0.1
west,widget a,4800,9,0.12
East,Widget B,,15,0.05
South,Widget C,2100,0,0.2
North,Widget A,4800,12,0.08
West,Widget A,5000,10,0.1
East,Widget A,7500,15,0.05
South,Widget C,1900,4,0.25
North,Widget A,5100,13,0.07
""")

print("=== STEP 1: SCAN ===")
r = scan_nulls_zeros(str(WORKFLOW_CSV))
print(f"Flagged columns: {r['flagged_columns']}")
for col, data in r["results"].items():
    print(f"  {col}: nulls={data['null_count']}, zeros={data.get('zero_count')}")
print(f"Suggested: {r['suggested_actions']}")

In [ ]:
print("\n=== STEP 2: VALIDATE ===")
r = validate_dataset(str(WORKFLOW_CSV))
print(f"Score: {r['score']}/100, Passed: {r['passed']}")
for issue in r["issues"]:
    print(f"  [{issue['severity']}] {issue['column']}: {issue['issue']}")

In [ ]:
print("\n=== STEP 3: CLEAN ===")
r = run_cleaning_pipeline(str(WORKFLOW_CSV), [
    {"op": "clean_text", "scope": "both"},
    {"op": "fill_nulls", "column": "Revenue", "strategy": "median"},
    {"op": "drop_duplicates"},
])
print(f"Applied {r['applied']} ops")
for s in r["summary"]:
    print(f"  {s['op']}")

In [ ]:
print("\n=== STEP 4: VERIFY ===")
r = validate_dataset(str(WORKFLOW_CSV))
print(f"New score: {r['score']}/100, Passed: {r['passed']}")
for issue in r["issues"]:
    print(f"  [{issue['severity']}] {issue['column']}: {issue['issue']}")

In [ ]:
print("\n=== STEP 5: AGGREGATE ===")
r = compute_aggregations(str(WORKFLOW_CSV), group_by=["Region"], agg_column="Revenue", agg_func="sum")
print(f"Total Revenue by Region:")
for row in r["result"]:
    print(f"  {row['Region']}: {row['Revenue']:,.0f}")

print("\nPASS: Full Tier 2 workflow complete")

## 08 — Cleanup

In [ ]:
# Uncomment to clean up test workspace
# shutil.rmtree(WORK)
# print("Workspace cleaned")

print(f"\nWorkspace preserved at: {WORK.absolute()}")
print("All Tier 2 tools tested successfully.")